# DG-TCAV Kaggle launcher
Enable a GPU and attach the processed Cohort A scans and trusted MedicalNet weights as private inputs. This notebook runs the classifier code. Preprocessing must be completed separately.

In [ ]:
!git clone https://github.com/Alishals28/dg-tcav-classifier-fyp.git
%cd dg-tcav-classifier-fyp
!pip install -r requirements.txt
!pip install -e . --no-deps

In [ ]:
import torch
print('PyTorch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
!git rev-parse HEAD
!python -m pytest

## Configure the inputs
Update `configs/kaggle.yaml` using the README: manifest, splits, scan directory, filename pattern, reference image, orientation, normalization and checkpoint path. Set `preprocessing_confirmed: true` after the preprocessing member confirms QC.

In [ ]:
# Check the delivered data before starting an experiment.
!python -m scripts.validate_dataset --config configs/kaggle.yaml
!python -m scripts.tiny_overfit --config configs/kaggle.yaml --output /kaggle/working/tiny_overfit

Before the full run, copy the config for a two-epoch trial with a separate experiment name. Keep the full-width model and `experiment.smoke_test: false`; set `training.epochs: 2`. Run that config to check GPU memory and epoch time, then use the main config below.

In [ ]:
!python -m src.train --config configs/kaggle.yaml

## Final evaluation and feature export
Replace `EXACT_RUN` with the directory printed by training. Uncomment test evaluation only after selecting and freezing the model using validation results. Save the entire run directory before the Kaggle session ends.

In [ ]:
# !python -m src.evaluate --config configs/kaggle.yaml --checkpoint /kaggle/working/outputs/EXACT_RUN/best.pt --split test --confirm-final-test
# !python -m src.export_activations --config configs/kaggle.yaml --checkpoint /kaggle/working/outputs/EXACT_RUN/best.pt --output /kaggle/working/features